# Antibiotic Stewardship & Patient Safety

**Antimicrobial stewardship rules (CR-7 through CR-11) in the CAP CDSS pipeline.**

> This notebook is generated by `_generate_stewardship_notebook.py`. Do not edit manually.

### Why Stewardship Matters

Antimicrobial resistance (AMR) is one of the top global public health threats. The WHO estimates
AMR directly caused 1.27 million deaths in 2019. Inappropriate antibiotic use in pneumonia drives
resistance — unnecessary broad-spectrum agents, failure to de-escalate when microbiology results
return, and avoidable fluoroquinolone exposure all contribute.

This notebook demonstrates the pipeline's **5 stewardship contradiction rules** (CR-7 through CR-11),
all resolved via **Strategy E** — deterministic Python logic with **0 GPU calls** for resolution.

### Rules Covered

| Rule | Name | Trigger |
|------|------|---------|
| CR-7 | Organism-antibiotic mismatch | Lab susceptibility or population resistance data contradicts current antibiotic |
| CR-8 | Unnecessary macrolide | Macrolide prescribed but no atypical pathogen on microbiology |
| CR-9 | IV-to-oral switch opportunity | IV >48h, patient stable, tolerating oral intake |
| CR-10 | Fluoroquinolone safety review | Fluoroquinolone prescribed for penicillin intolerance (not true allergy) |
| CR-11 | Pneumococcal de-escalation | Pneumococcal antigen positive + broad-spectrum antibiotic |

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **GITHUB_TOKEN** for private repo install
4. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 pandas>=2.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."
print("Authentication complete")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph
from cap_agent.agent.state import build_initial_state

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## Stewardship Rules Overview

All 5 stewardship rules use **Strategy E** — deterministic resolution with **0 GPU calls**.
The pipeline detects these contradictions in `check_contradictions` (CR-7, CR-8, CR-9, CR-11) or
`treatment_selection` (CR-10), then resolves them with pure Python logic.

| Rule | Detection | Condition | Resolution |
|------|-----------|-----------|------------|
| **CR-7** | `detect_contradictions` | Organism on micro has lab or population resistance to prescribed antibiotic | Flag mismatch, recommend susceptibility-guided switch |
| **CR-8** | `detect_contradictions` | Macrolide in `atypical_cover`, no atypical pathogen on micro, >=2 completed tests | Recommend macrolide de-escalation (exempt if high severity) |
| **CR-9** | `detect_contradictions` | IV >48h, all stability markers normal, oral tolerant, eating independently | Recommend IV-to-oral switch |
| **CR-10** | `treatment_selection` | Fluoroquinolone prescribed for penicillin intolerance (GI upset, not anaphylaxis) | Flag unnecessary FQ, recommend beta-lactam reassessment + MHRA warning |
| **CR-11** | `detect_contradictions` | Pneumococcal antigen positive + broad-spectrum antibiotic (co-amoxiclav/piperacillin) | Recommend de-escalation to amoxicillin |


## Load Stewardship Cases (Group C)

Group C contains 10 benchmark cases targeting CR-7 through CR-11. Each case has a `ground_truth`
dict specifying the expected contradiction rules that should fire.


In [ ]:
from benchmark_data.cases.registry import get_track2_cases

all_track2 = get_track2_cases()

# Group C: stewardship (case_id starts with "CR7-", "CR8-", "CR9-", "CR10-", "CR11-")
group_c = [c for c in all_track2 if any(c["case_id"].startswith(prefix) for prefix in ["CR7-", "CR8-", "CR9-", "CR10-", "CR11-"])]
print(f"Loaded {len(group_c)} stewardship cases")
for c in group_c:
    print(f"  {c['case_id']}: expected contradictions: {c['ground_truth'].get('contradictions', [])}")


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


## Run All Stewardship Cases

Run each Group C case through the full pipeline and collect results. These cases use mock
extraction (no CXR images, no FHIR bundles) so pipeline execution is fast — the focus is
on the deterministic stewardship logic.


In [ ]:
all_results = []

for i, case in enumerate(group_c):
    case_id = case["case_id"]
    print(f"[{i+1}/{len(group_c)}] Running {case_id}...")

    initial_state = build_initial_state(case)
    start = time.time()
    result = None
    for chunk in graph.stream(initial_state, stream_mode="values"):
        result = chunk
    elapsed = time.time() - start

    all_results.append({
        "case_id": case_id,
        "result": result,
        "elapsed": elapsed,
        "ground_truth": case["ground_truth"],
    })
    print(f"  Done in {elapsed:.1f}s — contradictions: "
          f"{[c['rule_id'] for c in result.get('contradictions_detected', [])]}")

print(f"\nAll {len(all_results)} cases complete")


## CR-7: Organism-Antibiotic Mismatch

**What it detects:** The prescribed antibiotic does not cover the identified organism based on
either laboratory susceptibility data or population-level resistance patterns.

**Cases:**
- **CR7-LAB-R:** S. pneumoniae resistant to amoxicillin on sputum culture susceptibility testing.
  The pipeline should flag the mismatch and recommend switching to a susceptible agent.
- **CR7-POP-R:** Legionella pneumophila identified on urinary antigen. Amoxicillin has no
  activity against Legionella (population-level resistance). The pipeline should flag the gap.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR7-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])
    abx = result.get("antibiotic_recommendation", {})
    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")
    print(f"\nAntibiotic: {abx.get('first_line', 'N/A')}")
    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")
    if resolutions:
        print(f"\nResolution (Strategy E):")
        for res in resolutions:
            print(f"  {res[:300]}")
    render_clinical_output(result, r["case_id"])


## CR-8: Unnecessary Macrolide

**What it detects:** A macrolide (e.g. clarithromycin) is prescribed as atypical cover, but
microbiology results show no atypical pathogen. With >=2 completed microbiological tests and
no atypical organism identified, the macrolide can be safely de-escalated.

**Important exemption:** High-severity CAP (CURB65 >= 3) is exempt from CR-8 because
clarithromycin is part of standard dual therapy (co-amoxiclav IV + clarithromycin), not
conditional atypical cover.

**Cases:**
- **CR8-FIRE:** Moderate severity (CURB65=2), S. pneumoniae on sputum + negative Legionella
  antigen. Macrolide should be flagged for de-escalation.
- **CR8-EXEMPT:** High severity (CURB65=3), same microbiology. CR-8 should NOT fire.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR8-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])
    abx = result.get("antibiotic_recommendation", {})
    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")
    print(f"\nAntibiotic: {abx.get('first_line', 'N/A')}")
    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")
    if resolutions:
        print(f"\nResolution (Strategy E):")
        for res in resolutions:
            print(f"  {res[:300]}")
    render_clinical_output(result, r["case_id"])


## CR-9: IV-to-Oral Switch Opportunity

**What it detects:** The patient has been on IV antibiotics for >48 hours but meets all
stability criteria for switching to oral therapy:
- Temperature < 37.5C
- Heart rate < 100 bpm
- Respiratory rate < 24/min
- SBP >= 90 mmHg
- SpO2 >= 90%
- No confusion (AMT >= 9)
- Tolerating oral intake and eating independently

Early IV-to-oral switch reduces length of stay and healthcare costs without compromising
clinical outcomes.

**Cases:**
- **CR9-MET:** All stability markers normal, 52h on IV, oral tolerant. CR-9 should fire.
- **CR9-NOT:** Temperature 38.2C (unstable). CR-9 should NOT fire.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR9-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])
    abx = result.get("antibiotic_recommendation", {})
    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")
    print(f"\nAntibiotic: {abx.get('first_line', 'N/A')}")
    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")
    if resolutions:
        print(f"\nResolution (Strategy E):")
        for res in resolutions:
            print(f"  {res[:300]}")
    render_clinical_output(result, r["case_id"])


## CR-10: Fluoroquinolone Safety Review

**What it detects:** A fluoroquinolone (levofloxacin) is prescribed because of a documented
penicillin "allergy", but the recorded reaction is actually an intolerance (e.g. GI upset,
nausea) rather than a true IgE-mediated allergy (anaphylaxis, urticaria, angioedema).

This is a critical patient safety rule:
- **MHRA safety warning:** Fluoroquinolones carry risks of tendon rupture, aortic dissection,
  and peripheral neuropathy — these are serious, sometimes irreversible adverse effects.
- Up to 90% of reported penicillin "allergies" are intolerances, not true allergies.
- Patients labelled with penicillin allergy receive more fluoroquinolones, more broad-spectrum
  agents, and have worse clinical outcomes.

**Cases:**
- **CR10-FIRE:** High severity + penicillin allergy (GI upset = intolerance). Levofloxacin
  prescribed. CR-10 should fire, recommending beta-lactam reassessment.
- **CR10-TRUE:** High severity + penicillin allergy (anaphylaxis = true allergy). Levofloxacin
  is appropriate. CR-10 should NOT fire.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR10-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])
    abx = result.get("antibiotic_recommendation", {})
    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")
    print(f"\nAntibiotic: {abx.get('first_line', 'N/A')}")
    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")
    if resolutions:
        print(f"\nResolution (Strategy E):")
        for res in resolutions:
            print(f"  {res[:300]}")
    render_clinical_output(result, r["case_id"])


## CR-11: Pneumococcal De-escalation

**What it detects:** A positive pneumococcal urinary antigen test confirms S. pneumoniae as the
causative organism, but the patient remains on a broad-spectrum antibiotic (co-amoxiclav or
piperacillin-tazobactam). Since pneumococcus is reliably susceptible to amoxicillin, the
pipeline recommends de-escalation to narrow-spectrum therapy.

**Cases:**
- **CR11-FIRE:** Pneumococcal antigen positive + co-amoxiclav IV (high severity). CR-11 should
  fire, recommending de-escalation to amoxicillin.
- **CR11-SUSC:** Same as above, but with explicit susceptibility data (amoxicillin: S).
  CR-11 should fire with high confidence.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR11-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])
    abx = result.get("antibiotic_recommendation", {})
    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")
    print(f"\nAntibiotic: {abx.get('first_line', 'N/A')}")
    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")
    if resolutions:
        print(f"\nResolution (Strategy E):")
        for res in resolutions:
            print(f"  {res[:300]}")
    render_clinical_output(result, r["case_id"])


## Stewardship Detection Accuracy

Per-rule recall and precision across all Group C cases. Recall measures whether the expected
contradiction was detected; precision measures whether detected contradictions are expected.


In [ ]:
import plotly.graph_objects as go

ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

# Collect all rule IDs
all_expected = set()
all_actual = set()
for r in all_results:
    gt = r["ground_truth"]
    all_expected |= set(gt.get("contradictions", []))
    all_actual |= {c["rule_id"] for c in r["result"].get("contradictions_detected", [])}

all_rules = sorted(all_expected | all_actual)

recalls = []
precisions = []
for rule in all_rules:
    tp = sum(1 for r in all_results
             if rule in {c["rule_id"] for c in r["result"].get("contradictions_detected", [])}
             and rule in set(r["ground_truth"].get("contradictions", [])))
    fn = sum(1 for r in all_results
             if rule in set(r["ground_truth"].get("contradictions", []))
             and rule not in {c["rule_id"] for c in r["result"].get("contradictions_detected", [])})
    fp = sum(1 for r in all_results
             if rule in {c["rule_id"] for c in r["result"].get("contradictions_detected", [])}
             and rule not in set(r["ground_truth"].get("contradictions", [])))
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)

# Print table
print(f"\n{B}{C}{'=' * 60}")
print(f"  STEWARDSHIP DETECTION ACCURACY")
print(f"{'=' * 60}{X}\n")

print(f"  {'Rule':<10} {'Recall':>10} {'Precision':>10}")
print(f"  {'─' * 30}")
for i, rule in enumerate(all_rules):
    rec_color = G if recalls[i] >= 1.0 else (Y if recalls[i] >= 0.5 else R)
    pre_color = G if precisions[i] >= 1.0 else (Y if precisions[i] >= 0.5 else R)
    print(f"  {rule:<10} {rec_color}{recalls[i]:>9.0%}{X} {pre_color}{precisions[i]:>9.0%}{X}")

# Plotly grouped bar chart
if all_rules:
    fig = go.Figure(data=[
        go.Bar(name="Recall", x=all_rules, y=recalls, marker_color="#2196F3"),
        go.Bar(name="Precision", x=all_rules, y=precisions, marker_color="#FF9800"),
    ])
    fig.update_layout(
        title="Stewardship Contradiction Detection (per rule)",
        barmode="group",
        yaxis=dict(range=[0, 1.05], title="Score"),
        width=600, height=400,
    )
    fig.show()


## Safety Score

Evaluate the safety evaluator across all stewardship cases. The safety score checks that
allergy contraindications are respected — critical for CR-10 (fluoroquinolone safety).


In [ ]:
from benchmark_data.evaluation.langsmith_evaluators import eval_safety, eval_antibiotic, eval_curb65

ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

print(f"\n{B}{C}{'=' * 60}")
print(f"  SAFETY & CLINICAL ACCURACY")
print(f"{'=' * 60}{X}\n")

print(f"  {'Case':<15} {'Safety':>8} {'Severity':>10} {'Antibiotic':>12}")
print(f"  {'─' * 45}")

safety_scores = []
severity_scores = []
abx_scores = []

for r in all_results:
    case_id = r["case_id"]
    result = r["result"]
    gt = r["ground_truth"]
    ref = {"ground_truth": gt}

    safety = eval_safety(result, ref)
    severity = eval_curb65(result, ref)
    antibiotic = eval_antibiotic(result, ref)

    s_score = safety["score"] if safety["score"] is not None else 1.0
    sev_score = severity["score"] if severity["score"] is not None else 1.0
    abx_score = antibiotic["score"] if antibiotic["score"] is not None else 1.0

    safety_scores.append(s_score)
    severity_scores.append(sev_score)
    abx_scores.append(abx_score)

    s_color = G if s_score >= 1.0 else R
    sev_color = G if sev_score >= 1.0 else R
    abx_color = G if abx_score >= 1.0 else R

    print(f"  {case_id:<15} {s_color}{s_score:>7.0%}{X} {sev_color}{sev_score:>9.0%}{X} {abx_color}{abx_score:>11.0%}{X}")

mean_safety = sum(safety_scores) / len(safety_scores) if safety_scores else 0
mean_severity = sum(severity_scores) / len(severity_scores) if severity_scores else 0
mean_abx = sum(abx_scores) / len(abx_scores) if abx_scores else 0

print(f"\n  {'Mean':<15} {mean_safety:>7.0%} {mean_severity:>9.0%} {mean_abx:>11.0%}")
print(f"\n{C}{'=' * 60}{X}")


## Verification Checklist

Expected vs detected contradictions per case — pass/fail summary.


In [ ]:
ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

print(f"\n{B}{C}{'=' * 70}")
print(f"  STEWARDSHIP VERIFICATION CHECKLIST")
print(f"{'=' * 70}{X}\n")

total_pass = 0
total_fail = 0

for r in all_results:
    case_id = r["case_id"]
    result = r["result"]
    gt = r["ground_truth"]
    expected = set(gt.get("contradictions", []))
    actual = {c["rule_id"] for c in result.get("contradictions_detected", [])}

    # Check: expected contradictions are detected
    missing = expected - actual
    # Check: no unexpected contradictions fired
    unexpected = actual - expected

    passed = len(missing) == 0 and len(unexpected) == 0
    status = f"{G}PASS{X}" if passed else f"{R}FAIL{X}"

    if passed:
        total_pass += 1
    else:
        total_fail += 1

    print(f"  [{status}] {case_id}")
    print(f"         Expected: {sorted(expected) if expected else 'none'}")
    print(f"         Detected: {sorted(actual) if actual else 'none'}")
    if missing:
        print(f"         {R}Missing: {sorted(missing)}{X}")
    if unexpected:
        print(f"         {Y}Unexpected: {sorted(unexpected)}{X}")

print(f"\n  {B}Results: {total_pass}/{total_pass + total_fail} cases passed{X}")
if total_fail == 0:
    print(f"  {G}All stewardship verification checks passed!{X}")
else:
    print(f"  {R}{total_fail} case(s) failed — review above for details{X}")

print(f"\n{C}{'=' * 70}{X}")
